In [68]:
import numpy as np
import pandas as pd
import yfinance as yf
import sys
sys.path.insert(0, "/Users/giovanafalcao/Desktop/Projects/trading-analytics")
from utils import yfinance_fix

TICKER = "AAPL"
BM_TICKER = "^GSPC" # Benchmark Ticker: S&P 500  
INTERVAL = "1d"
PERIOD = "730d" if INTERVAL == "1D" else "max"
SHIFT = 1
LOOKBACK = 10000

# Step 1: Fetch historical price (time-series) data, compute change, return and log return

def fetch_price_data(ticker = TICKER, interval = INTERVAL, period = PERIOD, shift = SHIFT, lookback = LOOKBACK):
    df = yf.download(ticker, session=yfinance_fix.chrome_session, interval = interval, period = period)
    df.columns = df.columns.get_level_values(0)
    df = df.reset_index(drop=True)   

    return df.iloc[-lookback:, :]

df = fetch_price_data()
df

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
1427,0.106418,0.109837,0.105991,0.109837,185203200
1428,0.108555,0.111546,0.106418,0.106418,173398400
1429,0.108127,0.110692,0.108127,0.108982,110140800
1430,0.114538,0.114538,0.108555,0.108982,183433600
1431,0.117103,0.117530,0.114111,0.114111,244160000
...,...,...,...,...,...
11422,260.480011,262.190002,259.019989,259.980011,31291500
11423,259.200012,260.179993,256.660004,259.730011,36234700
11424,258.829987,261.929993,257.190002,259.250000,48370700
11425,266.429993,266.559998,257.809998,258.160004,49735000


In [69]:
# Step 2: Fetch Benchmark data for stocks in the same sector or industry

def fetch_benchmark_data(ticker = BM_TICKER,  interval = INTERVAL, period = PERIOD, lookback = LOOKBACK):
    df_bm = yf.download(ticker, session=yfinance_fix.chrome_session, interval = interval, period = period)
    df_bm.columns = df_bm.columns.get_level_values(0)
    return df_bm.iloc[-lookback:, :]

df_bm = fetch_benchmark_data()
df_bm

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
1986-08-06,236.839996,237.350006,235.479996,237.029999,127500000
1986-08-07,237.039993,238.020004,236.309998,236.839996,122400000
1986-08-08,236.880005,238.059998,236.369995,237.039993,106300000
1986-08-11,240.679993,241.199997,236.869995,236.880005,125600000
1986-08-12,243.339996,243.369995,240.350006,240.679993,131700000
...,...,...,...,...,...
2026-04-10,6816.890137,6845.770020,6808.459961,6839.240234,4393220000
2026-04-13,6886.240234,6887.000000,6790.020020,6806.470215,4785840000
2026-04-14,6967.379883,6969.419922,6905.169922,6910.200195,5032380000


In [ ]:
# Step 3: Fetch fundamental (key-value) data 

def fetch_fundamental_data(ticker = TICKER):
    t = yf.Ticker(ticker)
    info = t.info
    return {
        "name": info.get("longName", ticker),
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "pe": info.get("trailingPE"),
        "forward_pe": info.get("forwardPE"),
        "market_cap": info.get("marketCap"),
        "revenue": info.get("totalRevenue"),
        "eps": info.get("trailingEps"),
        "dividend_yield": info.get("dividendYield"),
        "high_52w": info.get("fiftyTwoWeekHigh"),
        "low_52w": info.get("fiftyTwoWeekLow"),
        "beta": info.get("beta"),
        "profit_margin": info.get("profitMargins"),
        "roe": info.get("returnOnEquity"),
        "roa": info.get("returnOnAssets"),
        "debt_to_equity": info.get("debtToEquity"),
        "revenue_growth": info.get("revenueGrowth"),
        "gross_margins": info.get("grossMargins"),
        "current_ratio": info.get("currentRatio"),
        "price_to_book": info.get("priceToBook"),
        "ev_to_ebitda": info.get("enterpriseToEbitda"),
        "raw_info": info,
    }

dic_f = fetch_fundamental_data()
print(dic_f)
print(dic_f["name"])



{'name': 'Apple Inc.', 'sector': 'Technology', 'industry': 'Consumer Electronics', 'pe': 33.68268, 'forward_pe': 28.615034, 'market_cap': 3915968151552, 'revenue': 435617005568, 'eps': 7.91, 'dividend_yield': 0.39, 'high_52w': 288.62, 'low_52w': 189.81, 'beta': 1.109, 'profit_margin': 0.27037, 'roe': 1.5202099, 'roa': 0.24377, 'debt_to_equity': 102.63, 'revenue_growth': 0.157, 'gross_margins': 0.47325, 'current_ratio': 0.974, 'price_to_book': 44.419804, 'ev_to_ebitda': 25.736, 'raw_info': {'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'United States', 'phone': '(408) 996-1010', 'website': 'https://www.apple.com', 'industry': 'Consumer Electronics', 'industryKey': 'consumer-electronics', 'industryDisp': 'Consumer Electronics', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accesso

In [65]:
# Step 4: Fetch Alternative data 